In [1]:
# Cell 1
import boto3
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
# Cell 2
role = sagemaker.get_execution_role()
print(role)

arn:aws:iam::112949069548:role/LabRole


In [3]:
# Cell 3
s3 = boto3.client("s3")
response = s3.list_buckets()
for bucket in response['Buckets']:
    print(bucket['Name'])

sagemaker-us-east-1-112949069548


In [4]:
# Cell 4
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model/preprocessing.pkl", arcname="preprocessing.pkl")
    tar.add("model/best_model.pkl", arcname="best_model.pkl")

In [5]:
# Cell 5 - upload ke S3
import boto3

s3 = boto3.client("s3")
bucket_name = "sagemaker-us-east-1-112949069548"   # sesuaikan dengan nama bucket

s3.upload_file("model.tar.gz", bucket_name, "model/model_credit.tar.gz")

In [8]:
# Cell 6 - sanity check: model bisa di-load lewat kontrak SageMaker (model_fn)
from inference import model_fn
model = model_fn("model")

In [9]:
# Cell 7
import os
import shutil

os.makedirs("code", exist_ok=True)
shutil.copy("inference.py", "code/inference.py")
shutil.copy("preprocessing.py", "code/preprocessing.py")

print("Isi folder code/:", os.listdir("code"))

Isi folder code/: ['inference.py', 'preprocessing.py']


In [10]:
# Cell 8
import json
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

BUCKET = "sagemaker-us-east-1-112949069548"   # sesuaikan dengan nama bucket
MODEL_S3_KEY = "model/model_credit.tar.gz"
ENDPOINT_NAME = "credit-score-endpoint"

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"   # match scikit-learn==1.4.2 yang dipin di train.ipynb

def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]

def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        source_dir="code",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )

    print("Testing endpoint with sample data...")
    sample = {
        "instances": [{
            "Age": 35, "Annual_Income": 20364.57, "Monthly_Inhand_Salary": 1626.05,
            "Num_Bank_Accounts": 6, "Num_Credit_Card": 8, "Interest_Rate": 32,
            "Num_of_Loan": 3, "Delay_from_due_date": 23, "Num_of_Delayed_Payment": 9,
            "Changed_Credit_Limit": 10.3, "Num_Credit_Inquiries": 11, "Outstanding_Debt": 2500.04,
            "Credit_Utilization_Ratio": 27.58, "Credit_History_Age": 116, "Total_EMI_per_month": 26.17,
            "Amount_invested_monthly": 92.52, "Monthly_Balance": 303.92,
            "Credit_Mix": "Standard", "Month": "August", "Occupation": "Journalist",
            "Payment_of_Min_Amount": "Yes", "Payment_Behaviour": "High_spent_Small_value_payments",
            "Auto_Loan": 0, "Credit_Builder_Loan": 0, "Debt_Consolidation_Loan": 0,
            "Home_Equity_Loan": 0, "Mortgage_Loan": 1, "Payday_Loan": 0,
            "Personal_Loan": 1, "Student_Loan": 1
        }]
    }

    runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample),
    )
    print("\nSmoke test response:")
    print(response["Body"].read().decode("utf-8"))

if __name__ == "__main__":
    main()

Role:      arn:aws:iam::112949069548:role/LabRole
Model URI: s3://sagemaker-us-east-1-112949069548/model/model_credit.tar.gz
Endpoint:  credit-score-endpoint

Deploying endpoint (5-8 minutes)...
------!Testing endpoint with sample data...

Smoke test response:
[{"prediction": "Standard", "probabilities": {"Good": 0.0012338254686166624, "Poor": 0.4896037529190058, "Standard": 0.5091624216123777}}]


In [9]:
# Cell 9 (cleanup kalau deploy gagal/stuck)
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-score-endpoint"

print(f"Deleting endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete!")

Deleting endpoint: credit-score-endpoint...
Endpoint deletion triggered.
Deleting endpoint configuration: credit-score-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete!
